# 01 - Generate CARLA Data

This notebook generates synthetic multi-task training data from CARLA 0.9.15 on a local Ubuntu machine.

Paper reference implemented in this notebook:
- Simulated Autonomous Driving: Integrating Deep Learning and Computer Vision for Self-Driving Cars (ResearchGate, 2026).

## Section 1 - Setup
Install local dependencies and import modules.

In [ ]:
%pip install -q carla==0.9.15 opencv-python pandas numpy pillow tqdm

In [ ]:
from pathlib import Path
from queue import Queue, Empty
import random
import time
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import carla

## Section 2 - Paths and Configuration
Define output folders, collection limits, and CARLA settings.

In [ ]:
repo_root = Path.cwd().resolve().parents[0] if (Path.cwd() / 'notebooks').exists() else Path.cwd().resolve()
data_root = repo_root / 'data' / 'carla'
images_dir = data_root / 'images'
labels_dir = data_root / 'labels'
masks_dir = data_root / 'masks'
for p in (images_dir, labels_dir, masks_dir):
    p.mkdir(parents=True, exist_ok=True)

manifest_path = data_root / 'manifest.csv'
target_frames = 800
image_w, image_h = 640, 640
fov = 90
towns = ['Town01', 'Town02', 'Town03']
weather_presets = [
    carla.WeatherParameters.ClearNoon,
    carla.WeatherParameters.CloudyNoon,
    carla.WeatherParameters.WetNoon,
    carla.WeatherParameters.HardRainNoon,
]

# CARLA semantic IDs: 6=RoadLine (lane), 7=Road (drivable)
LANE_CLASS_ID = 6
DRIVABLE_CLASS_ID = 7
DOMAIN_LABEL = 0

print(f'Repo root: {repo_root}')
print(f'Output dir: {data_root}')

## Section 3 - Helper Functions
Connect to CARLA, spawn actors, convert labels, and write files.

In [ ]:
def connect_client(host='127.0.0.1', port=2000, timeout=10.0):
    client = carla.Client(host, port)
    client.set_timeout(timeout)
    return client


def spawn_ego_vehicle(world):
    bp_lib = world.get_blueprint_library()
    vehicle_bps = bp_lib.filter('vehicle.*')
    spawn_points = world.get_map().get_spawn_points()
    if not spawn_points:
        raise RuntimeError('No spawn points available in current town.')
    for _ in range(20):
        vehicle_bp = random.choice(vehicle_bps)
        spawn_point = random.choice(spawn_points)
        vehicle = world.try_spawn_actor(vehicle_bp, spawn_point)
        if vehicle is not None:
            vehicle.set_autopilot(True)
            return vehicle
    raise RuntimeError('Failed to spawn ego vehicle after multiple attempts.')


def spawn_camera(world, vehicle, sensor_type, image_size=(640, 640), fov=90):
    bp_lib = world.get_blueprint_library()
    cam_bp = bp_lib.find(sensor_type)
    cam_bp.set_attribute('image_size_x', str(image_size[0]))
    cam_bp.set_attribute('image_size_y', str(image_size[1]))
    cam_bp.set_attribute('fov', str(fov))
    cam_tf = carla.Transform(carla.Location(x=1.5, z=2.4))
    return world.spawn_actor(cam_bp, cam_tf, attach_to=vehicle)


def image_to_bgr_array(image):
    arr = np.frombuffer(image.raw_data, dtype=np.uint8).reshape((image.height, image.width, 4))
    return arr[:, :, :3]


def seg_to_masks(seg_image, lane_class=6, road_class=7):
    # Semantic labels are in channel 2 for raw CityScapes palette output.
    seg = np.frombuffer(seg_image.raw_data, dtype=np.uint8).reshape((seg_image.height, seg_image.width, 4))
    semantic_id = seg[:, :, 2]
    lane_mask = (semantic_id == lane_class).astype(np.uint8) * 255
    drivable_mask = (semantic_id == road_class).astype(np.uint8) * 255
    return lane_mask, drivable_mask


def project_bbox_to_yolo(actor, world_to_camera, camera, img_w, img_h):
    bbox = actor.bounding_box
    vertices = bbox.get_world_vertices(actor.get_transform())

    points = []
    fov = float(camera.attributes['fov'])
    focal = img_w / (2.0 * np.tan(np.deg2rad(fov) / 2.0))

    for v in vertices:
        p = np.array([v.x, v.y, v.z, 1.0], dtype=np.float32)
        cam = world_to_camera @ p
        z = cam[2]
        if z <= 0.1:
            continue

        u = (focal * cam[0] / z) + img_w / 2.0
        v_img = (focal * -cam[1] / z) + img_h / 2.0
        points.append((u, v_img))

    if len(points) < 2:
        return None

    xs = np.clip([pt[0] for pt in points], 0, img_w - 1)
    ys = np.clip([pt[1] for pt in points], 0, img_h - 1)
    x_min, x_max = float(np.min(xs)), float(np.max(xs))
    y_min, y_max = float(np.min(ys)), float(np.max(ys))

    w = x_max - x_min
    h = y_max - y_min
    if w < 4 or h < 4:
        return None

    cx = (x_min + x_max) * 0.5 / img_w
    cy = (y_min + y_max) * 0.5 / img_h
    nw = w / img_w
    nh = h / img_h
    return cx, cy, nw, nh


def save_yolo_labels(world, camera, path, img_w, img_h):
    ego_location = camera.get_transform().location
    world_to_camera = np.array(camera.get_transform().get_inverse_matrix(), dtype=np.float32)

    lines = []
    for actor in world.get_actors().filter('vehicle.*'):
        if actor.id == camera.parent.id:
            continue
        if actor.get_location().distance(ego_location) > 60.0:
            continue
        yolo_box = project_bbox_to_yolo(actor, world_to_camera, camera, img_w, img_h)
        if yolo_box is None:
            continue
        cx, cy, nw, nh = yolo_box
        lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

    path.write_text('\n'.join(lines), encoding='utf-8')



## Section 4 - Data Collection Loop
Collect 800 frames across mixed towns and weather, then save images, labels, masks, and manifest entries.

In [ ]:
client = connect_client()
rows = []
frame_idx = 0

for town in towns:
    if frame_idx >= target_frames:
        break

    world = client.load_world(town)
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.05
    world.apply_settings(settings)

    traffic_manager = client.get_trafficmanager()
    traffic_manager.set_synchronous_mode(True)

    actors = []
    try:
        vehicle = spawn_ego_vehicle(world)
        rgb_cam = spawn_camera(world, vehicle, 'sensor.camera.rgb', (image_w, image_h), fov)
        seg_cam = spawn_camera(world, vehicle, 'sensor.camera.semantic_segmentation', (image_w, image_h), fov)
        actors.extend([vehicle, rgb_cam, seg_cam])

        rgb_queue = Queue()
        seg_queue = Queue()
        rgb_cam.listen(rgb_queue.put)
        seg_cam.listen(seg_queue.put)

        weather_cycle = weather_presets.copy()
        random.shuffle(weather_cycle)

        with tqdm(total=target_frames, initial=frame_idx, desc=f'Collecting {town}') as pbar:
            while frame_idx < target_frames:
                weather = weather_cycle[frame_idx % len(weather_cycle)]
                world.set_weather(weather)

                world.tick()
                try:
                    rgb_image = rgb_queue.get(timeout=1.0)
                    seg_image = seg_queue.get(timeout=1.0)
                except Empty:
                    continue

                stem = f'frame_{frame_idx:05d}'
                img_path = images_dir / f'{stem}.jpg'
                lbl_path = labels_dir / f'{stem}.txt'
                lane_path = masks_dir / f'lane_{frame_idx:05d}.png'
                drivable_path = masks_dir / f'drivable_{frame_idx:05d}.png'

                rgb = image_to_bgr_array(rgb_image)
                lane_mask, drivable_mask = seg_to_masks(seg_image, LANE_CLASS_ID, DRIVABLE_CLASS_ID)

                cv2.imwrite(str(img_path), rgb)
                cv2.imwrite(str(lane_path), lane_mask)
                cv2.imwrite(str(drivable_path), drivable_mask)
                save_yolo_labels(world, rgb_cam, lbl_path, image_w, image_h)

                rows.append({
                    'image_path': str(img_path),
                    'label_path': str(lbl_path),
                    'lane_mask_path': str(lane_path),
                    'drivable_mask_path': str(drivable_path),
                    'domain': DOMAIN_LABEL,
                    'town': town,
                    'weather': weather.__class__.__name__,
                })

                frame_idx += 1
                pbar.n = frame_idx
                pbar.refresh()

                if frame_idx >= target_frames:
                    break

    finally:
        for actor in actors:
            try:
                actor.stop() if hasattr(actor, 'stop') else None
            except Exception:
                pass
            try:
                actor.destroy()
            except Exception:
                pass

        settings = world.get_settings()
        settings.synchronous_mode = False
        settings.fixed_delta_seconds = None
        world.apply_settings(settings)
        traffic_manager.set_synchronous_mode(False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print(f'Saved {len(rows)} samples to {manifest_path}')